# EmberWatch — M2 smoke detector: data → study → training (Kaggle GPU)

Owner: Mouhib · Sprint 1 · CDC target: **smoke mAP@0.5 ≥ 0.55** (stretch ≥ 0.65) on the test split.

**Before *Run all* (right panel):**
1. *Settings → Accelerator*: **GPU T4 x2** (or P100) · *Internet*: **On** (needs a phone-verified account).
2. *Add-ons → Secrets*: add **`ROBOFLOW_API_KEY`** (roboflow.com → Settings → API key) and attach it to this notebook.
3. *Add data* (optional, faster): search **smoke-fire-detection-yolo** (D-Fire). If not attached, it is downloaded automatically.
4. Keep the notebook **Private** (confidential project).

In [ ]:
# 1) GPU check + code (branch m2-detection) + extra libraries
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q -b m2-detection https://github.com/achrefchaabene/EmberWatch.git /kaggle/working/EmberWatch
%cd /kaggle/working/EmberWatch
!pip install -q ultralytics sahi roboflow kagglehub albumentations

In [ ]:
# 2) Download the datasets listed in configs/datasets.yaml (skipped if already present)
import json
RAW = '/kaggle/working/data/raw'
!python -m src.detection.download --only dfire hpwren_smoke --out {RAW}
manifest = json.load(open(f'{RAW}/manifest.json'))
DFIRE, HPWREN = manifest['dfire'], manifest['hpwren_smoke']
print('D-Fire :', DFIRE); print('HPWREN :', HPWREN)

## 3) Study the data BEFORE choosing filters and hyperparameters
Each number below drives one decision (see `docs/M2_DONNEES_ET_ENTRAINEMENT.md`).

In [ ]:
# 3a) EDA of each raw source (5,000-image uniform sample = a few minutes)
!python -m src.detection.eda --data {DFIRE}  --out artifacts/eda_dfire_raw  --max-images 5000
!python -m src.detection.eda --data {HPWREN} --out artifacts/eda_hpwren_raw --max-images 5000

In [ ]:
# 3b) Look at the figures (box sizes, where smoke is, image quality)
from IPython.display import Image as Show, Markdown, display
for name in ('dfire', 'hpwren'):
    d = f'artifacts/eda_{name}_raw'
    display(Markdown(open(f'{d}/eda.md').read().split('![')[0]))
    for fig in ('box_sizes', 'box_centres', 'image_quality'):
        display(Show(f'{d}/{fig}.png'))

In [ ]:
# 4) Turn the study into settings (printed, so every choice is visible in the notebook)
import yaml
eda_d = json.load(open('artifacts/eda_dfire_raw/eda.json'))
eda_h = json.load(open('artifacts/eda_hpwren_raw/eda.json'))
cfg = yaml.safe_load(open('configs/m2_yolo.yaml'))

# Grayscale: the TARGET domain is the tower cameras (HPWREN), so its share sets ToGray.
cfg['albumentations']['to_gray'] = max(0.05, round(eda_h['grayscale_share'], 2))
# Blur augmentation only if the data is not already blurry.
if max(eda_d['blurry_share'], eda_h['blurry_share']) >= 0.2:
    cfg['albumentations']['blur'] = 0.0
# Offline resize only for very large images (loading speed); boxes are normalised so they stay valid.
MAX_SIDE = 1280 if max(eda_d['image_long_side']['median'], eda_h['image_long_side']['median']) > 1600 else None

yaml.safe_dump(cfg, open('configs/m2_yolo_kaggle.yaml', 'w'), sort_keys=False)
print('ToGray p =', cfg['albumentations']['to_gray'], '| blur p =', cfg['albumentations']['blur'], '| MAX_SIDE =', MAX_SIDE)
print('Small boxes: D-Fire', eda_d['boxes']['small_share'], '| HPWREN', eda_h['boxes']['small_share'])

In [ ]:
# 5) Build the clean, leakage-free smoke dataset (drops corrupt images, invalid boxes, cross-split copies)
OUT = '/kaggle/working/m2_smoke'
extra = f'--max-side {MAX_SIDE}' if MAX_SIDE else ''
!python -m src.detection.prepare_dataset --dfire {DFIRE} --hpwren {HPWREN} --out {OUT} {extra}
report = json.load(open(f'{OUT}/split_report.json'))
print(json.dumps({k: report[k] for k in ('sources', 'splits', 'removed_cross_split_duplicates')}, indent=1))
assert report['leakage']['leakage_free'], 'Leakage: do not train'

In [ ]:
# 6) Second leakage check on content: near-duplicates across splits (open the listed files if > 0)
!python -m src.detection.eda --data {OUT} --out artifacts/eda_m2_processed --max-images 0
eda_p = json.load(open('artifacts/eda_m2_processed/eda.json'))
print('near-duplicate groups across splits:', eda_p['near_duplicate_groups_across_splits'])
print(eda_p['near_duplicate_examples'][:5])

In [ ]:
# 7) 3-minute sanity run (5 % of images, 1 epoch): catches errors before spending hours of GPU
!python -m src.detection.train --data {OUT}/data.yaml --config configs/m2_yolo_kaggle.yaml --epochs 1 --fraction 0.05 --device 0 --name sanity

In [ ]:
# 8) Full training (≈ several hours; early stopping + checkpoint every 5 epochs)
!python -m src.detection.train --data {OUT}/data.yaml --config configs/m2_yolo_kaggle.yaml --device 0 --name yolo_smoke

**If the session stops (12 h limit / disconnection):** *Save Version* first so `runs/` is kept, then in a new session re-run cells 1–2 and:

```
!python -m src.detection.train --resume runs/m2/yolo_smoke/weights/last.pt
```

In [ ]:
# 9) Evaluate on the TEST split + latency with / without SAHI -> artifacts/m2_metrics.json
!python -m src.detection.evaluate --weights runs/m2/yolo_smoke/weights/best.pt --data {OUT}/data.yaml --device 0
print(open('artifacts/m2_metrics.json').read())

In [ ]:
# 10) Visual check: tiled predictions on test frames (look for misses and false alarms)
import numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from src.detection.detector import SmokeDetector
det = SmokeDetector('runs/m2/yolo_smoke/weights/best.pt', device='cuda:0')
imgs = sorted((Path(OUT) / 'images' / 'test').glob('hpwren__*'))[:3] + sorted((Path(OUT) / 'images' / 'test').glob('dfire__*'))[:3]
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, p in zip(axes.flat, imgs):
    frame = np.asarray(Image.open(p).convert('RGB'))
    ax.imshow(frame); ax.set_title(p.name[:40]); ax.axis('off')
    for d in det.predict(frame):
        x1, y1, x2, y2 = d.box
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='orange', lw=2))
        ax.text(x1, y1 - 4, f'{d.confidence:.2f}', color='orange')
plt.tight_layout(); plt.savefig('artifacts/m2_test_predictions.png', dpi=80)

In [ ]:
# 11) Files to download from the Output tab (weights are NOT pushed to git)
import shutil
for src, dst in [('runs/m2/yolo_smoke/weights/best.pt', 'm2_smoke_best.pt'),
                 ('runs/m2/yolo_smoke/run_config.json', 'm2_run_config.json'),
                 ('artifacts/m2_metrics.json', 'm2_metrics.json'),
                 (f'{OUT}/split_report.json', 'm2_split_report.json'),
                 ('configs/m2_yolo_kaggle.yaml', 'm2_yolo_kaggle.yaml')]:
    shutil.copy(src, f'/kaggle/working/{dst}')
shutil.make_archive('/kaggle/working/m2_eda', 'zip', 'artifacts')
print('Download m2_smoke_best.pt -> put it in artifacts/ to serve it with the API.')